# 

In [1]:
import argparse
import logging
import os
import pathlib as pl
import sys
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import tifffile
import timm
import torch
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/scgpt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### remapping mouse to human names

In [1]:
import scanpy as sc
adata = sc.read_h5ad("/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/data/adata_annotated.h5ad")
adata.var_names_make_unique()
print(adata)
print(adata.var_names[:10])  # what do the gene names look like?

AnnData object with n_obs × n_vars = 113277 × 19059
    obs: 'orig.ident', 'nCount_Spatial', 'nFeature_Spatial', 'Region', 'alignment_clusters', 'spot_class', 'first_type', 'second_type', 'min_score', 'singlet_score', 'seurat_cluster.sketched', 'seurat_clusters', 'metaclusters', 'AnatomicalSite', 'WoundSite', 'seurat_cluster.projected', 'seurat_cluster.projected.score', 'leverage.score', 'banksy_cluster_0.8', 'banksy_cluster_0.5', 'barcode'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'
Index(['Xkr4', 'Rp1', 'Sox17', 'Lypla1', 'Tcea1', 'Rgs20', 'Atp6v1h', 'Oprk1',
       'Npbwr1', 'Rb1cc1'],
      dtype='object')


/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/spatialfusion/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [3]:
human_adata.var_names

Index(['XKR4', 'RP1', 'SOX17', 'LYPLA1', 'TCEA1', 'RGS20', 'ATP6V1H', 'OPRK1',
       'NPBWR1', 'RB1CC1',
       ...
       'MT-CO2', 'MT-ATP8', 'MT-ATP6', 'MT-CO3', 'MT-ND3', 'MT-ND4L', 'MT-ND4',
       'MT-ND5', 'MT-ND6', 'VAMP7'],
      dtype='object', length=15562)

In [2]:
import json

human_adata = adata.copy() 

# load scGPT vocab
with open("/insomnia001/depts/morpheus/users/me2982/models/scgpt/vocab.json") as f:
    vocab = json.load(f)
vocab_genes = set(vocab.keys())

# uppercase mouse gene names → human convention
human_adata.var_names = [g.upper() for g in human_adata.var_names]

# keep only genes in scGPT vocab
genes_to_keep = [g for g in human_adata.var_names if g in vocab_genes]
human_adata = human_adata[:, genes_to_keep].copy()

print(f"Genes retained: {len(genes_to_keep)}")
print(f"Final adata: {human_adata}")

# save
human_adata.write_h5ad("/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/data/adata_humanized.h5ad")

Genes retained: 15562
Final adata: AnnData object with n_obs × n_vars = 113277 × 15562
    obs: 'orig.ident', 'nCount_Spatial', 'nFeature_Spatial', 'Region', 'alignment_clusters', 'spot_class', 'first_type', 'second_type', 'min_score', 'singlet_score', 'seurat_cluster.sketched', 'seurat_clusters', 'metaclusters', 'AnatomicalSite', 'WoundSite', 'seurat_cluster.projected', 'seurat_cluster.projected.score', 'leverage.score', 'banksy_cluster_0.8', 'banksy_cluster_0.5', 'barcode'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'


In [2]:
import sys
import logging
sys.path.append('/insomnia001/depts/morpheus/users/me2982/models/scgpt/zero-shot-scfoundation/')

from sc_foundation_evals import scgpt_forward, data
from sc_foundation_evals.helpers.custom_logging import log

log.setLevel(logging.INFO)

/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/scgpt_env/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/scgpt_env/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


In [3]:
def run_embed_scGPT(
    dataset_path: str,
    model_dir: str,  # path to the pre-trained model, 3 files are expected: model_weights (best_model.pt), model args (args.json), and model vocab (vocab.json)
    output_dir: str,  # output_dir is the path to which the results should be saved
    n_hvg: int,
    gene_col: str = "index",  # in which column in adata.obs are gene names stored? if they are in index, the index will be copied to a column with this name
    layer_key: str = "X",  # where the raw counts are stored?
    log_norm: bool = False,  # are the values log_norm already?
    seed: int = 42,
    max_seq_len: int = 1200,  # maximum sequence of the input is controlled by max_seq_len, here I'm using the pretrained default
    batch_size: int = 32,  # batch_size depends on available GPU memory; should be a multiple of 8
    input_bins: int = 51,
    model_run: str = "pretrained",
    num_workers: int = 0,  # if you can use multithreading specify num_workers
    output_name: str = "scGPT.parquet",
    scfoundation_dir: str | None = None,
) -> None:
    if scfoundation_dir:
        sys.path.append(scfoundation_dir)
    else:
        sys.path.append(str(DEFAULT_SCFOUNDATION_DIR))

    from sc_foundation_evals import scgpt_forward, data
    from sc_foundation_evals.helpers.custom_logging import log

    log.setLevel(logging.INFO)


    # create the model
    scgpt_model = scgpt_forward.scGPT_instance(
        saved_model_path=model_dir,
        model_run=model_run,
        batch_size=batch_size,
        save_dir=output_dir,
        num_workers=num_workers,
        explicit_save_dir=True,
    )

    # create config
    scgpt_model.create_configs(seed=seed, max_seq_len=max_seq_len, n_bins=input_bins)

    scgpt_model.load_pretrained_model()

    # This is to keep the model running if the amount of genes is small
    input_data = data.InputData(adata_dataset_path=dataset_path)
    vocab_list = scgpt_model.vocab.get_stoi().keys()

    adata = input_data.adata
    genes_in_vocab = adata.var_names.intersection(vocab_list)
    if len(genes_in_vocab) / len(adata.var_names) < 0.5:
        log.warning("Fewer than 50% of genes are found in the model vocab — continuing anyway.")
    
    adata._inplace_subset_var(genes_in_vocab)
    input_data.adata = adata

    input_data.preprocess_data(
        gene_vocab=vocab_list,
        model_type="scGPT",
        gene_col=gene_col,
        data_is_raw=not log_norm,
        counts_layer=layer_key,
        n_bins=input_bins,
        n_hvg=n_hvg,
    )

    scgpt_model.tokenize_data(
        data=input_data, input_layer_key="X_binned", include_zero_genes=False
    )

    scgpt_model.extract_embeddings(data=input_data)

    pd.DataFrame(
        input_data.adata.obsm["X_scGPT"],
        index=input_data.adata.obs["cell_id"] if "cell_id" in input_data.adata.obs.columns else input_data.adata.obs.index
    ).to_parquet(pl.Path(output_dir) / output_name)


In [4]:
run_embed_scGPT(
    dataset_path = "/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/data/adata_humanized.h5ad",
    model_dir = "/insomnia001/depts/morpheus/users/me2982/models/scgpt",
    output_dir = "/insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/",
    n_hvg=1200,
    gene_col="index",
    layer_key="X",
    log_norm=False,
    seed=42,
    max_seq_len=1200,
    batch_size=16,
    input_bins=51,
    model_run="pretrained",
    num_workers=0,
    output_name="scGPT.parquet",
    scfoundation_dir="/insomnia001/depts/morpheus/users/me2982/models/scgpt/zero-shot-scfoundation/"

)

INFO     | 2026-06-09 07:08:41 | Using device cuda
WARNING  | 2026-06-09 07:08:41 | Overriding pre-trained config['save_dir'] with /insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/ (was /scratch/ssd004/datasets/cellxgene/save/cellxgene_census_human-May23-08-36-2023)
WARNING  | 2026-06-09 07:08:41 | Overriding pre-trained config['max_seq_len'] with 1200 (was 1200)
INFO     | 2026-06-09 07:08:41 | Loading vocab from /insomnia001/depts/morpheus/users/me2982/models/scgpt/vocab.json


INFO     | 2026-06-09 07:08:42 | Loading model from /insomnia001/depts/morpheus/users/me2982/models/scgpt/best_model.pt
WARNING  | 2026-06-09 07:08:42 | Loading partial model params from /insomnia001/depts/morpheus/users/me2982/models/scgpt/best_model.pt
WARNING  | 2026-06-09 07:08:42 | Cannot load transformer_encoder.layers.0.self_attn.in_proj_weight with shape torch.Size([1536, 512])
WARNING  | 2026-06-09 07:08:42 | Cannot load transformer_encoder.layers.0.self_attn.in_proj_bias with shape torch.Size([1536])
WARNING  | 2026-06-09 07:08:42 | Cannot load transformer_encoder.layers.1.self_attn.in_proj_weight with shape torch.Size([1536, 512])
WARNING  | 2026-06-09 07:08:42 | Cannot load transformer_encoder.layers.1.self_attn.in_proj_bias with shape torch.Size([1536])
WARNING  | 2026-06-09 07:08:42 | Cannot load transformer_encoder.layers.2.self_attn.in_proj_weight with shape torch.Size([1536, 512])
WARNING  | 2026-06-09 07:08:42 | Cannot load transformer_encoder.layers.2.self_attn.in_pr

scGPT - INFO - Filtering genes by counts ...
scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Log1p transforming ...
scGPT - INFO - Subsetting highly variable genes ...
scGPT - WARNING - No batch_key is provided, will use all cells for HVG selection.
scGPT - INFO - Binning data ...
scGPT - WARNING - The input data contains all zero rows. Please make sure this is expected. You can use the `filter_cell_by_counts` arg to filter out all zero rows.
scGPT - WARNING - The input data contains all zero rows. Please make sure this is expected. You can use the `filter_cell_by_counts` arg to filter out all zero rows.
scGPT - WARNING - The input data contains all zero rows. Please make sure this is expected. You can use the `filter_cell_by_counts` arg to filter out all zero rows.
scGPT - WARNING - The input data contains all zero rows. Please make sure this is expected. You can use the `filter_cell_by_counts` arg to filter out all zero rows.
scGPT - WARNING - The input data contains all z

INFO     | 2026-06-09 07:09:08 | Tokenizing data
INFO     | 2026-06-09 07:09:13 | Preparing dataloader
INFO     | 2026-06-09 07:09:13 | Saving config to /insomnia001/depts/morpheus/users/me2982/data/unwounded_woappi/
INFO     | 2026-06-09 07:09:13 | Extracting embeddings
INFO     | 2026-06-09 07:09:14 | Extracting embeddings for batch 1/7080
INFO     | 2026-06-09 07:09:22 | Extracting embeddings for batch 709/7080
INFO     | 2026-06-09 07:09:30 | Extracting embeddings for batch 1417/7080
INFO     | 2026-06-09 07:09:38 | Extracting embeddings for batch 2125/7080
INFO     | 2026-06-09 07:09:46 | Extracting embeddings for batch 2833/7080
INFO     | 2026-06-09 07:09:54 | Extracting embeddings for batch 3541/7080
INFO     | 2026-06-09 07:10:01 | Extracting embeddings for batch 4249/7080
INFO     | 2026-06-09 07:10:09 | Extracting embeddings for batch 4957/7080
INFO     | 2026-06-09 07:10:17 | Extracting embeddings for batch 5665/7080
INFO     | 2026-06-09 07:10:25 | Extracting embeddings fo